## 00. Quick Start


In [1]:
print('Concept Portfolio V2 Lab — staged와 one-click은 동일 Core를 사용합니다.')
print('LIVE_TEST_LEVEL로 CORE / LEGAL_C1 / FULL_E2E / ONE_CLICK 중 하나만 선택하세요.')

Concept Portfolio V2 Lab — staged와 one-click은 동일 Core를 사용합니다.
LIVE_TEST_LEVEL로 CORE / LEGAL_C1 / FULL_E2E / ONE_CLICK 중 하나만 선택하세요.


## 01. Environment


In [2]:
import os, sys, json
from pathlib import Path
from IPython.display import display
SEARCH_ROOTS = [Path.cwd(), *Path.cwd().parents]
AI_ROOT = next((p for p in SEARCH_ROOTS if (p / 'app').is_dir()), None)
if AI_ROOT is None: AI_ROOT = next((p / 'ai' for p in SEARCH_ROOTS if (p / 'ai' / 'app').is_dir()))
if str(AI_ROOT) not in sys.path: sys.path.insert(0, str(AI_ROOT))
print({'python': sys.version.split()[0], 'aiRoot': str(AI_ROOT)})

{'python': '3.14.5', 'aiRoot': 'C:\\Users\\seewo\\Desktop\\big_proj_01\\new_3\\ai'}


## 02. MODE


In [3]:
MODE = 'LIVE'  # MOCK | REPLAY | LIVE
RECORDINGS_DIR = AI_ROOT / 'recordings' / 'concept_portfolio_v2'
print({'mode': MODE, 'liveExternalOperationsEnabled': MODE == 'LIVE'})

{'mode': 'LIVE', 'liveExternalOperationsEnabled': True}


## 03. Environment Check


In [4]:
LIVE_ENV_KEYS = ['AI_PROVIDER', 'AI_API_KEY', 'AI_MODEL', 'MOLEG_API_KEY', 'LEGAL_REGISTRY_VERSION']
env_status = {key: bool(os.getenv(key)) for key in LIVE_ENV_KEYS}
print(env_status if MODE == 'LIVE' else {'mode': MODE, 'message': '외부 환경변수 불필요'})

{'AI_PROVIDER': True, 'AI_API_KEY': True, 'AI_MODEL': True, 'MOLEG_API_KEY': True, 'LEGAL_REGISTRY_VERSION': True}


## 04. Schema Preflight


In [5]:
from app.concept_portfolio_v2 import ConceptPortfolioEngine, ProviderGateway, ProviderMode
from app.concept_portfolio_v2.adapters import CurrentLegalAdapter
from app.concept_portfolio_v2.diagnostics.notebook_view import *
gateway = ProviderGateway(MODE, recordings_dir=RECORDINGS_DIR)
engine = ConceptPortfolioEngine(MODE, gateway=gateway)
schema_preflight = engine.schema_preflight_report()
display(show_schema_preflight(schema_preflight))
assert schema_preflight.status == 'PASS' and schema_preflight.providerCalls == 0

,스키마,상태,실패,Provider 호출
0,PlanDraftPool,PASS,[],0
1,ConceptCandidateDraft,PASS,[],0
2,SemanticDistinctnessResult,PASS,[],0
3,SemanticFidelityResult,PASS,[],0
4,SemanticArchitectureBatch,PASS,[],0
5,SemanticHypothesisBatch,PASS,[],0
6,BusinessRoleSemanticBatch,PASS,[],0
7,LegalFactDependencySemanticBatch,PASS,[],0
8,LegalFactCompletionPatch,PASS,[],0


## 05. Input


In [6]:
SCENARIO_FILE = AI_ROOT / 'fixtures' / 'concept_portfolio_v2' / 'live_scenarios.json'
SCENARIOS = {item['scenarioId']: item for item in json.loads(SCENARIO_FILE.read_text(encoding='utf-8'))}
LIVE_SCENARIO = 'FOOD_PHYSICAL_COMMERCE'
LIVE_TEST_LEVEL = 'FULL_E2E'  # CORE | LEGAL_C1 | FULL_E2E | ONE_CLICK
RUN_STAGED_CORE = LIVE_TEST_LEVEL in {'CORE', 'LEGAL_C1', 'FULL_E2E'}
RUN_STAGED_LEGAL = LIVE_TEST_LEVEL in {'LEGAL_C1', 'FULL_E2E'}
RUN_STAGED_FULL = LIVE_TEST_LEVEL == 'FULL_E2E'
scenario = SCENARIOS[LIVE_SCENARIO]
TEST_INPUT = {key: scenario[key] for key in ('ideaOverview', 'problem', 'targetUsers')}
MAX_CONCEPTS = 5
display({'scenario': LIVE_SCENARIO, 'testLevel': LIVE_TEST_LEVEL, 'domain': scenario['domain'],
         'expectedStructuralFeatures': scenario['expectedStructuralFeatures']})

{'scenario': 'FOOD_PHYSICAL_COMMERCE',
 'testLevel': 'FULL_E2E',
 'domain': 'Food physical commerce',
 'expectedStructuralFeatures': ['물리적 이행', '직접 운영과 파트너 역할 구분']}

## 06. Idea Brief Derivation


In [7]:
seed = engine.seed_adapter.adapt(TEST_INPUT)
idea_context = None
if RUN_STAGED_CORE:
    engine._reset()
    idea_context = await engine.derive_idea_brief(seed)
print({'ideaCallComplete': bool(idea_context), 'interpretationPresent': bool(seed.interpretation),
       'stagedCore': RUN_STAGED_CORE})

{'ideaCallComplete': True, 'interpretationPresent': True, 'stagedCore': True}


## 07. Safety


In [8]:
display(idea_context.safetyReview.model_dump(mode='json') if idea_context else {'status': 'SKIPPED'})
assert idea_context is None or idea_context.safetyReview.passed

{'decision': 'ALLOW',
 'categories': [],
 'restrictions': [],
 'userFacingReason': '이 아이디어는 안전하며, 개인 맞춤형 식재료 제공과 관련된 문제를 해결하는 데 기여할 수 있습니다.'}

## 08. AI가 이해한 아이디어


In [9]:
display(show_idea_interpretation(idea_context) if idea_context else {'status': 'SKIPPED'})

,항목,AI 이해 결과
0,interpretedProblem,1~2인 가구에서 발생하는 음식물 쓰레기 문제를 해결하기 위한 서비스.
1,interpretedTargetUsers,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구.
2,usageContext,소량의 식재료를 제공하고 남은 재료를 활용할 수 있는 레시피를 안내하는 서비스.
3,industryCategory,식품 및 요리 서비스
4,researchScope,소량 식재료 제공 및 음식물 쓰레기 감소 관련 시장.
5,conciseIdeaDefinition,개인 맞춤형 소량 식재료와 활용 레시피를 제공하는 서비스.
6,targetRegionInterpretation,대상 지역은 명시되지 않음.
7,relevantKnownCompetitorContext,경쟁자는 명시되지 않음.


## 09. Readiness / Summary / commitments


In [10]:
display(show_idea_readiness(idea_context) if idea_context else {'status': 'SKIPPED'})

{'readiness': {'status': 'READY_FOR_REVIEW',
  'score': 0,
  'missingFieldKeys': []},
 'readinessDiagnostic': 'READINESS_INCONSISTENT',
 'userFacingSummary': '이 서비스는 개인 맞춤형 소량 식재료를 제공하고, 남은 재료를 활용한 레시피를 안내하여 1~2인 가구의 음식물 쓰레기 문제를 해결하는 것을 목표로 합니다.',
 'commitmentCandidates': [],
 'contradictions': [],
 'questions': []}

## 10. Seed Analysis


In [11]:
analysis = await engine.analyze_seed(seed) if RUN_STAGED_CORE else None
display(show_seed_analysis(analysis) if analysis else {'status': 'SKIPPED'})

,구분,값
0,탐색 폭,EXPLORE
1,다양성 수용량,5
2,설명,선택 입력 LOCK 0개로 11개 설계 차원이 열려 있습니다. diversityCa...


## 11. Generic Opportunity Kernel


In [12]:
display(analysis.opportunityKernel.model_dump(mode='json') if analysis else {'status': 'SKIPPED'})

{'problemCore': '1~2인 가구에서 발생하는 음식물 쓰레기 문제를 해결하기 위한 서비스.',
 'targetCore': '요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구.',
 'useContexts': ['소량의 식재료를 제공하고 남은 재료를 활용할 수 있는 레시피를 안내하는 서비스.'],
 'intentComponents': ['개인 맞춤형 소량 식재료와 활용 레시피를 제공하는 서비스.'],
 'mustPreserve': ['1~2인 가구에서 발생하는 음식물 쓰레기 문제를 해결하기 위한 서비스.',
  '요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구.',
  '개인 맞춤형 소량 식재료와 활용 레시피를 제공하는 서비스.'],
 'maySpecialize': ['핵심 대상의 의미 있는 하위 세그먼트',
  '핵심 사용 맥락의 구체화',
  '가치 제안 또는 offer의 구체화'],
 'forbiddenDriftSummary': '핵심 문제와 대상이 모두 무관한 기회로 교체되면 범위를 벗어납니다.'}

## 12. Design Space


In [13]:
display(show_design_space(analysis) if analysis else {'status': 'SKIPPED'})

,분류,필드,값
0,SOURCE_LOCK,ideaOverview,개인 맞춤형 소량 식재료를 제공하고 남은 재료 활용 레시피를 안내하는 서비스
1,SOURCE_LOCK,problem,1~2인 가구가 큰 포장 단위 때문에 식재료를 남기고 음식물 쓰레기가 발생한다
2,SOURCE_LOCK,targetUsers,요리할 시간이 적고 낭비를 줄이고 싶은 1~2인 가구
3,SEMANTIC_ANCHOR,ideaOverview,개인 맞춤형 소량 식재료를 제공하고 남은 재료 활용 레시피를 안내하는 서비스
4,SEMANTIC_ANCHOR,problem,1~2인 가구가 큰 포장 단위 때문에 식재료를 남기고 음식물 쓰레기가 발생한다
5,SEMANTIC_ANCHOR,targetUsers,요리할 시간이 적고 낭비를 줄이고 싶은 1~2인 가구
6,OPEN,solutionMechanism,변경 가능
7,OPEN,valueDelivery,변경 가능
8,OPEN,operatingModel,변경 가능
9,OPEN,supplyStructure,변경 가능


## 13. Generate and Adaptively Replenish Plan Pool


In [14]:
plan_validation = (await engine.prepare_portfolio_plans(seed, analysis, max_concepts=MAX_CONCEPTS)
                   if RUN_STAGED_CORE else None)
plans = engine._last_plan_pool if plan_validation else []
print({'totalPlans': len(plans),
       'planningRounds': plan_validation.planningRounds if plan_validation else 0,
       'replenishmentRequested': plan_validation.replenishmentRequested if plan_validation else 0})

{'totalPlans': 6, 'planningRounds': 1, 'replenishmentRequested': 0}


## 14. Plan Count / Adaptive Replenishment Check


In [15]:
display(show_plan_pool_status(engine._last_plan_pool_status) if plan_validation else {'status': 'SKIPPED'})
display({'planningRounds': plan_validation.planningRounds if plan_validation else 0,
         'replenishmentRequested': plan_validation.replenishmentRequested if plan_validation else 0,
         'adaptiveReplenishmentUsed': bool(plan_validation and plan_validation.planningRounds > 1)})

,requestedPoolSize,returnedPoolSize,initialTarget,reserveTarget,reserveAvailable,status
0,7,6,5,2,1,RESERVE_SHORTFALL


{'planningRounds': 1,
 'replenishmentRequested': 0,
 'adaptiveReplenishmentUsed': False}

## 15. Korean Plan Display


In [16]:
display(show_portfolio_plans(plan_validation.acceptedPlans + plan_validation.reservePlans)
        if plan_validation else {'status': 'SKIPPED'})

,planId,제목,선택 상태,selectionScore,selectionReason,relationToPortfolio,Concept Family,Target Thesis,Use Context,Value Thesis,Offer Thesis,Solution Thesis,Architecture,비교 가치
0,P1,소량 맞춤형 식재료 서비스,SELECTED,0.8072,Opportunity fit과 Concept clarity가 가장 높은 대표안,PORTFOLIO_SEED,기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구,소량의 식재료를 제공하고 남은 재료를 활용할 수 있는 레시피를 안내하는 서비스,필요한 만큼의 식재료를 제공하여 음식물 쓰레기를 줄이고 요리를 간편하게 만든다,소량의 식재료와 맞춤형 레시피를 통해 요리의 즐거움을 제공한다,"식재료를 소량으로 제공하여 낭비를 줄이고, 레시피를 통해 요리의 효율성을 높인다","{'businessRole': 'OTHER', 'operatingModel': 'O...",1~2인 가구의 요리 문제를 해결하기 위한 혁신적인 접근 방식.
1,P3,맞춤형 레시피 추천 서비스,SELECTED,0.8054,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구,소량의 식재료를 활용할 수 있는 레시피를 추천하는 서비스,고객의 취향에 맞춘 레시피를 제공하여 요리의 즐거움을 더한다,개인 맞춤형 레시피로 요리의 효율성을 높인다,소량의 식재료를 활용한 레시피를 통해 음식물 쓰레기를 줄인다,"{'businessRole': 'OTHER', 'operatingModel': 'O...",고객의 요리 경험을 개선하고 음식물 쓰레기를 줄이는 서비스.
2,P2,간편 요리 키트 서비스,SELECTED,0.6095,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,요리할 시간이 부족한 1~2인 가구,소량의 식재료와 간편한 조리법을 제공하는 서비스,"간편한 요리 키트를 통해 요리의 부담을 줄이고, 식재료 낭비를 방지한다",간편한 조리법과 소량 식재료로 요리를 쉽게 만든다,"요리 키트를 통해 요리 시간을 단축하고, 남은 재료를 활용할 수 있도록 돕는다","{'businessRole': 'OTHER', 'operatingModel': 'O...",요리의 간편함과 식재료 낭비 문제를 동시에 해결하는 서비스.
3,P4,신선한 재료 정기 배송 서비스,SELECTED,0.5130,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구,소량의 신선한 재료를 정기적으로 배송받는 서비스,신선한 재료를 정기적으로 제공하여 요리의 편리함을 더한다,정기 배송으로 신선한 재료를 제공하여 요리의 질을 높인다,소량의 신선한 재료를 통해 음식물 쓰레기를 줄인다,"{'businessRole': 'OTHER', 'operatingModel': 'O...",신선한 재료를 통해 요리의 질을 높이고 음식물 쓰레기를 줄이는 서비스.
4,P6,식재료 관리 및 레시피 서비스,SELECTED,0.4290,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구,식재료 관리와 함께 소량의 레시피를 제공하는 서비스,"식재료 관리를 통해 고객의 요리 효율성을 높이고, 낭비를 줄인다",식재료 관리와 레시피 제공으로 요리의 편리함을 더한다,소량의 식재료를 활용한 관리 서비스를 통해 음식물 쓰레기를 줄인다,"{'businessRole': 'OTHER', 'operatingModel': 'O...",고객의 식재료 관리를 통해 요리 효율성을 높이고 음식물 쓰레기를 줄이는 서비스.
5,P5,요리 교육 및 레시피 서비스,RESERVE,0.6690,Selected Portfolio 대비 marginal value가 낮아 reser...,"DISTINCT,VARIANT",기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구,요리 교육과 함께 소량의 식재료를 제공하는 서비스,"요리 교육을 통해 고객의 요리 능력을 향상시키고, 식재료 낭비를 줄인다",요리 교육과 레시피 제공으로 요리의 즐거움을 더한다,소량의 식재료를 활용한 요리 교육을 통해 음식물 쓰레기를 줄인다,"{'businessRole': 'OTHER', 'operatingModel': 'O...",고객의 요리 능력을 향상시키고 음식물 쓰레기를 줄이는 서비스.


## 16. Plan Lock/Intent Validation


In [17]:
display({'accepted': [p.planId for p in plan_validation.acceptedPlans] if plan_validation else [],
         'rejected': [p.model_dump(mode='json') for p in plan_validation.rejectedPlans]
                     if plan_validation else []})

{'accepted': ['P1', 'P3', 'P2', 'P4', 'P6'], 'rejected': []}

## 17. Portfolio Family / Variant / Distinct


In [18]:
display(show_plan_diversity(plan_validation.diversity) if plan_validation else {'status': 'SKIPPED'})

,A,B,판정,Family A,Family B,겹침,실질 차이,단계,semantic judge,관계 설명
0,P1,P2,DISTINCT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, de...","targetSegmentThesis, useCaseThesis, valuePropo...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
1,P1,P3,DISTINCT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, tr...","deliveryModel, useCaseThesis, valueProposition...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
2,P2,P3,DISTINCT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, tr...","deliveryModel, targetSegmentThesis, useCaseThe...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
3,P1,P4,DISTINCT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, de...","useCaseThesis, valuePropositionThesis, offerTh...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
4,P2,P4,DISTINCT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, de...","targetSegmentThesis, useCaseThesis, valuePropo...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
5,P3,P4,VARIANT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, tr...","deliveryModel, useCaseThesis, valueProposition...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
6,P1,P5,DISTINCT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, transactionModel...","partnerModel, deliveryModel, useCaseThesis, va...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
7,P2,P5,DISTINCT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, transactionModel...","partnerModel, deliveryModel, targetSegmentThes...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
8,P3,P5,VARIANT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, deliveryModel, t...","partnerModel, useCaseThesis, valuePropositionT...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
9,P4,P5,VARIANT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, transactionModel...","partnerModel, deliveryModel, useCaseThesis, va...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...


## 18. Selected + Reserve Plans


In [19]:
selected_plans = plan_validation.acceptedPlans if plan_validation else []
reserve_plans = plan_validation.reservePlans if plan_validation else []
display(show_portfolio_plans(selected_plans + reserve_plans))
display({'selected': [p.planId for p in selected_plans], 'reserve': [p.planId for p in reserve_plans]})

,planId,제목,선택 상태,selectionScore,selectionReason,relationToPortfolio,Concept Family,Target Thesis,Use Context,Value Thesis,Offer Thesis,Solution Thesis,Architecture,비교 가치
0,P1,소량 맞춤형 식재료 서비스,SELECTED,0.8072,Opportunity fit과 Concept clarity가 가장 높은 대표안,PORTFOLIO_SEED,기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구,소량의 식재료를 제공하고 남은 재료를 활용할 수 있는 레시피를 안내하는 서비스,필요한 만큼의 식재료를 제공하여 음식물 쓰레기를 줄이고 요리를 간편하게 만든다,소량의 식재료와 맞춤형 레시피를 통해 요리의 즐거움을 제공한다,"식재료를 소량으로 제공하여 낭비를 줄이고, 레시피를 통해 요리의 효율성을 높인다","{'businessRole': 'OTHER', 'operatingModel': 'O...",1~2인 가구의 요리 문제를 해결하기 위한 혁신적인 접근 방식.
1,P3,맞춤형 레시피 추천 서비스,SELECTED,0.8054,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구,소량의 식재료를 활용할 수 있는 레시피를 추천하는 서비스,고객의 취향에 맞춘 레시피를 제공하여 요리의 즐거움을 더한다,개인 맞춤형 레시피로 요리의 효율성을 높인다,소량의 식재료를 활용한 레시피를 통해 음식물 쓰레기를 줄인다,"{'businessRole': 'OTHER', 'operatingModel': 'O...",고객의 요리 경험을 개선하고 음식물 쓰레기를 줄이는 서비스.
2,P2,간편 요리 키트 서비스,SELECTED,0.6095,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,요리할 시간이 부족한 1~2인 가구,소량의 식재료와 간편한 조리법을 제공하는 서비스,"간편한 요리 키트를 통해 요리의 부담을 줄이고, 식재료 낭비를 방지한다",간편한 조리법과 소량 식재료로 요리를 쉽게 만든다,"요리 키트를 통해 요리 시간을 단축하고, 남은 재료를 활용할 수 있도록 돕는다","{'businessRole': 'OTHER', 'operatingModel': 'O...",요리의 간편함과 식재료 낭비 문제를 동시에 해결하는 서비스.
3,P4,신선한 재료 정기 배송 서비스,SELECTED,0.5130,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구,소량의 신선한 재료를 정기적으로 배송받는 서비스,신선한 재료를 정기적으로 제공하여 요리의 편리함을 더한다,정기 배송으로 신선한 재료를 제공하여 요리의 질을 높인다,소량의 신선한 재료를 통해 음식물 쓰레기를 줄인다,"{'businessRole': 'OTHER', 'operatingModel': 'O...",신선한 재료를 통해 요리의 질을 높이고 음식물 쓰레기를 줄이는 서비스.
4,P6,식재료 관리 및 레시피 서비스,SELECTED,0.4290,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구,식재료 관리와 함께 소량의 레시피를 제공하는 서비스,"식재료 관리를 통해 고객의 요리 효율성을 높이고, 낭비를 줄인다",식재료 관리와 레시피 제공으로 요리의 편리함을 더한다,소량의 식재료를 활용한 관리 서비스를 통해 음식물 쓰레기를 줄인다,"{'businessRole': 'OTHER', 'operatingModel': 'O...",고객의 식재료 관리를 통해 요리 효율성을 높이고 음식물 쓰레기를 줄이는 서비스.
5,P5,요리 교육 및 레시피 서비스,RESERVE,0.6690,Selected Portfolio 대비 marginal value가 낮아 reser...,"DISTINCT,VARIANT",기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구,요리 교육과 함께 소량의 식재료를 제공하는 서비스,"요리 교육을 통해 고객의 요리 능력을 향상시키고, 식재료 낭비를 줄인다",요리 교육과 레시피 제공으로 요리의 즐거움을 더한다,소량의 식재료를 활용한 요리 교육을 통해 음식물 쓰레기를 줄인다,"{'businessRole': 'OTHER', 'operatingModel': 'O...",고객의 요리 능력을 향상시키고 음식물 쓰레기를 줄이는 서비스.


{'selected': ['P1', 'P3', 'P2', 'P4', 'P6'], 'reserve': ['P5']}

## 19. Candidate 1


In [20]:
candidate_one = (await engine.expand_plan(seed, selected_plans[0], 1)
                 if RUN_STAGED_CORE and selected_plans else None)
display(show_candidates([candidate_one]) if candidate_one else [])

,candidateId,lineageId,parentCandidateId,이름,핵심 작동방식,family,descriptor,수익,운영
0,C1,L1,None,소량 맞춤형 식재료 서비스,"식재료를 소량으로 제공하여 낭비를 줄이고, 레시피를 통해 요리의 효율성을 높인다.",거래 중개 · 기타 운영,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 모델을 통해 정기적으로 수익을 창출한다.,주문 후 24시간 이내에 신선한 식재료를 배송한다.


## 20. Candidate 1 Korean/Governance


In [21]:
candidate_one_reports = []
display({'candidateId': candidate_one.candidateId if candidate_one else None,
         'status': 'PENDING_FULL_CANDIDATE_RECOVERY'})

{'candidateId': 'C1', 'status': 'PENDING_FULL_CANDIDATE_RECOVERY'}

## 21. Candidate 1 Actual Generic Descriptor


In [22]:
display(show_concept_descriptors([candidate_one]) if candidate_one else [])

,entityId,family,dimension,code,confidence,source
0,C1,INTERMEDIARY:OTHER,businessRole,INTERMEDIARY,HIGH,RULE
1,C1,INTERMEDIARY:OTHER,operatingModel,OTHER,LOW,UNKNOWN
2,C1,INTERMEDIARY:OTHER,partnerModel,PARTNER_NETWORK,HIGH,RULE
3,C1,INTERMEDIARY:OTHER,deliveryModel,PHYSICAL_DELIVERY,HIGH,RULE
4,C1,INTERMEDIARY:OTHER,transactionModel,OTHER,LOW,UNKNOWN
5,C1,INTERMEDIARY:OTHER,monetizationModel,OTHER,LOW,UNKNOWN
6,C1,INTERMEDIARY:OTHER,customerInteractionModel,OTHER,LOW,UNKNOWN
7,C1,INTERMEDIARY:OTHER,dataDependency,NONE,NaN,NaN
8,C1,INTERMEDIARY:OTHER,physicalDependency,MATERIAL,NaN,NaN


## 22. Candidate 1 Fidelity


In [23]:
display({'candidateId': candidate_one.candidateId if candidate_one else None,
         'fidelity': '전체 Candidate Recovery 단계에서 semantic fallback 포함 검증'})

{'candidateId': 'C1',
 'fidelity': '전체 Candidate Recovery 단계에서 semantic fallback 포함 검증'}

## 23. Remaining Candidates


In [24]:
remaining_candidates = []
if RUN_STAGED_CORE:
    for i, plan in enumerate(selected_plans[1:], 2):
        remaining_candidates.append(await engine.expand_plan(seed, plan, i))
candidate_drafts = ([candidate_one] if candidate_one else []) + remaining_candidates
display(show_candidates(candidate_drafts))

,candidateId,lineageId,parentCandidateId,이름,핵심 작동방식,family,descriptor,수익,운영
0,C1,L1,None,소량 맞춤형 식재료 서비스,"식재료를 소량으로 제공하여 낭비를 줄이고, 레시피를 통해 요리의 효율성을 높인다.",거래 중개 · 기타 운영,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 모델을 통해 정기적으로 수익을 창출한다.,주문 후 24시간 이내에 신선한 식재료를 배송한다.
1,C2,L2,None,맞춤형 레시피 추천 서비스,고객의 취향과 재료를 기반으로 한 레시피 추천 시스템을 통해 소량의 식재료를 활용한...,거래 중개 · 파트너 네트워크,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 모델을 통한 수익 창출,"온라인 플랫폼을 통해 레시피를 추천하고, 고객의 피드백을 반영하여 지속적으로 개선한다."
2,C3,L3,None,간편 요리 키트 서비스,"요리 키트를 통해 요리 시간을 단축하고, 남은 재료를 활용할 수 있도록 돕는다.",기타 역할 · 기타 운영,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 모델로 지속적인 수익을 창출한다,"온라인 플랫폼을 통해 주문을 받고, 물류 시스템으로 신선한 키트를 배송한다."
3,C4,L4,None,신선한 재료 정기 배송 서비스,정기 배송 시스템을 통한 신선한 재료 제공으로 고객의 요리 주기에 맞춰 신선한 재료...,거래 중개 · 파트너 네트워크,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 모델을 통한 지속적인 수익 창출,"고객의 피드백을 반영하여 재료의 품질을 개선하고, 지역 농가와 협력하여 신선한 재료..."
4,C5,L5,None,식재료 관리 및 레시피 서비스,소량의 식재료를 활용한 관리 서비스를 통해 음식물 쓰레기를 줄인다.,기타 역할 · 기타 운영,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 모델,온라인 플랫폼을 통해 식재료 관리 서비스를 운영한다.


## 24. Candidate Actual Generic Descriptors


In [25]:
display(show_concept_descriptors(candidate_drafts))

,entityId,family,dimension,code,confidence,source
0,C1,INTERMEDIARY:OTHER,businessRole,INTERMEDIARY,HIGH,RULE
1,C1,INTERMEDIARY:OTHER,operatingModel,OTHER,LOW,UNKNOWN
2,C1,INTERMEDIARY:OTHER,partnerModel,PARTNER_NETWORK,HIGH,RULE
3,C1,INTERMEDIARY:OTHER,deliveryModel,PHYSICAL_DELIVERY,HIGH,RULE
4,C1,INTERMEDIARY:OTHER,transactionModel,OTHER,LOW,UNKNOWN
5,C1,INTERMEDIARY:OTHER,monetizationModel,OTHER,LOW,UNKNOWN
6,C1,INTERMEDIARY:OTHER,customerInteractionModel,OTHER,LOW,UNKNOWN
7,C1,INTERMEDIARY:OTHER,dataDependency,NONE,NaN,NaN
8,C1,INTERMEDIARY:OTHER,physicalDependency,MATERIAL,NaN,NaN
9,C2,INTERMEDIARY:PARTNER_NETWORK,businessRole,INTERMEDIARY,HIGH,RULE


## 25. Candidate Recovery / Portfolio Relations


In [26]:
candidate_preparation = (await engine.prepare_candidate_portfolio(
    seed, plan_validation, max_concepts=MAX_CONCEPTS, initial_candidates=candidate_drafts)
    if RUN_STAGED_CORE and plan_validation else None)
candidates = candidate_preparation.candidates if candidate_preparation else []
candidate_reports = candidate_preparation.reports if candidate_preparation else []
display(show_candidate_recovery(candidate_preparation) if candidate_preparation else {'status': 'SKIPPED'})
candidate_pairwise = [engine.compare_candidates(candidates[i], candidates[j])
                      for i in range(len(candidates)) for j in range(i + 1, len(candidates))]
display(show_plan_diversity(candidate_pairwise))

{'summary':    candidateGenerated  candidateAcceptedInitially  candidateRegenerated  \
 0                   5                           5                     0   
 
    candidateRecovered  reservePlansActivated  candidateRecoveryReplans  \
 0                   0                      0                         0   
 
    finalCandidatePortfolio  
 0                        5  ,
 'attempts':   candidateId  schemaValid  hardLockPreserved  semanticAnchorPreserved  \
 0          C1         True               True                     True   
 1          C2         True               True                     True   
 2          C3         True               True                     True   
 3          C4         True               True                     True   
 4          C5         True               True                     True   
 
    planFidelity anchorDecision fidelityDecision  contentLanguageValid  \
 0          True           PASS          ADAPTED                  True   
 1        

,A,B,판정,Family A,Family B,겹침,실질 차이,단계,semantic judge,관계 설명
0,C1,C2,DISTINCT,INTERMEDIARY:OTHER,INTERMEDIARY:PARTNER_NETWORK,"businessRole, partnerModel, customerInteractio...","operatingModel, transactionModel, monetization...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
1,C1,C3,DISTINCT,INTERMEDIARY:OTHER,OTHER:OTHER,"operatingModel, partnerModel, transactionModel...","businessRole, monetizationModel, deliveryModel...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
2,C1,C4,DISTINCT,INTERMEDIARY:OTHER,INTERMEDIARY:PARTNER_NETWORK,"businessRole, partnerModel, deliveryModel, tra...","operatingModel, monetizationModel, valuePropos...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
3,C1,C5,DISTINCT,INTERMEDIARY:OTHER,OTHER:OTHER,"operatingModel, deliveryModel, customerInterac...","businessRole, partnerModel, transactionModel, ...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
4,C2,C3,DISTINCT,INTERMEDIARY:PARTNER_NETWORK,OTHER:OTHER,"partnerModel, deliveryModel, monetizationModel...","businessRole, operatingModel, transactionModel...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
5,C2,C4,DISTINCT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, mo...","transactionModel, deliveryModel, dataDependenc...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
6,C2,C5,DISTINCT,INTERMEDIARY:PARTNER_NETWORK,OTHER:OTHER,"transactionModel, monetizationModel, customerI...","businessRole, operatingModel, partnerModel, de...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
7,C3,C4,DISTINCT,OTHER:OTHER,INTERMEDIARY:PARTNER_NETWORK,"partnerModel, transactionModel, monetizationMo...","businessRole, operatingModel, deliveryModel, c...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
8,C3,C5,DISTINCT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, monetizationMode...","partnerModel, transactionModel, deliveryModel,...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
9,C4,C5,DISTINCT,INTERMEDIARY:PARTNER_NETWORK,OTHER:OTHER,"deliveryModel, monetizationModel, customerInte...","businessRole, operatingModel, partnerModel, tr...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.


## 26. Legal Fact Completeness + Business Design Completion + C1 Fact Pattern


In [27]:
prechecks = [engine.legal_precheck(item) for item in candidates] if RUN_STAGED_LEGAL else []
display(show_legal_precheck(prechecks))
legal_preparation = (await engine.prepare_legal_candidates(seed, candidates)
                     if RUN_STAGED_LEGAL and candidates else None)
candidates_before_legal = candidates
candidates = legal_preparation.candidates if legal_preparation else candidates
display({'factCompleteness': [item.model_dump(mode='json') for item in legal_preparation.reports]
                              if legal_preparation else [],
         'roleSemanticBatchCalls': legal_preparation.roleSemanticBatchCalls if legal_preparation else 0,
         'dependencySemanticBatchCalls': legal_preparation.dependencySemanticBatchCalls if legal_preparation else 0,
         'completionAttempted': legal_preparation.completionAttempted if legal_preparation else 0,
         'completionValidated': legal_preparation.completionValidated if legal_preparation else 0,
         'completionAccepted': legal_preparation.completionAccepted if legal_preparation else 0,
         'completionExhausted': legal_preparation.completionExhausted if legal_preparation else 0,
         'completionCompliance': [item.model_dump(mode='json') for item in legal_preparation.completionCompliance] if legal_preparation else [],
         'factConsistency': [item.model_dump(mode='json') for item in legal_preparation.consistencyReports] if legal_preparation else [],
         'consistencyRepairAttempted': legal_preparation.consistencyRepairAttempted if legal_preparation else 0,
         'consistencyRepairAccepted': legal_preparation.consistencyRepairAccepted if legal_preparation else 0,
         'consistencyRepairExhausted': legal_preparation.consistencyRepairExhausted if legal_preparation else 0,
         'preLegalExclusions': legal_preparation.excludedCandidates if legal_preparation else []})
display(show_legal_fact_pattern(candidates[0].candidate, seed) if candidates else [])

,candidateId,label,directSeller,intermediary,regulatedPhysicalActivity,personalDataDependency,qualificationDependency,riskHints
0,C1,Structural risk precheck — not final legal review,True,True,True,False,False,[물리 활동]
1,C2,Structural risk precheck — not final legal review,True,False,True,True,False,"[물리 활동, 개인정보]"
2,C3,Structural risk precheck — not final legal review,True,False,True,False,False,[물리 활동]
3,C4,Structural risk precheck — not final legal review,True,True,True,False,False,[물리 활동]
4,C5,Structural risk precheck — not final legal review,True,False,True,True,False,"[물리 활동, 개인정보]"


{'factCompleteness': [{'candidateId': 'C1',
   'status': 'COMPLETE',
   'missingDesignFacts': [],
   'contradictions': [],
   'completionRequirements': [],
   'affectedFields': [],
   'roleSemantics': [{'field': 'platformRole',
     'status': 'MATCH',
     'presence': 'PRESENT',
     'deterministicStatus': 'MATCH',
     'semanticUsed': False,
     'semanticStatus': 'NOT_RUN',
     'finalStatus': 'MATCH',
     'safeReason': '역할 의미가 필드와 일치합니다.'},
    {'field': 'providerRole',
     'status': 'MATCH',
     'presence': 'PRESENT',
     'deterministicStatus': 'MATCH',
     'semanticUsed': False,
     'semanticStatus': 'NOT_RUN',
     'finalStatus': 'MATCH',
     'safeReason': '역할 의미가 필드와 일치합니다.'},
    {'field': 'sellerRole',
     'status': 'MATCH',
     'presence': 'PRESENT',
     'deterministicStatus': 'MATCH',
     'semanticUsed': False,
     'semanticStatus': 'NOT_RUN',
     'finalStatus': 'MATCH',
     'safeReason': '역할 의미가 필드와 일치합니다.'},
    {'field': 'intermediaryRole',
     'status': 'M

,field,legalFact,source,authority,decision
0,platformRole,"온라인 플랫폼을 통해 주문을 받고, 물류 시스템을 통해 신선한 식재료를 배송한다.",CONCEPT_GENERATED,REVIEWABLE,PROPOSED
1,providerRole,식재료 공급 및 배송을 담당하는 업체,CONCEPT_GENERATED,REVIEWABLE,PROPOSED
2,sellerRole,고객에게 식재료를 판매하는 역할,CONCEPT_GENERATED,REVIEWABLE,PROPOSED
3,intermediaryRole,물류 및 배송을 담당하는 중개업체,CONCEPT_GENERATED,REVIEWABLE,PROPOSED
4,transactionFlow,"[고객이 앱을 통해 주문, 주문 정보가 물류업체에 전달, 물류업체가 식재료를 준비 ...",CONCEPT_GENERATED,REVIEWABLE,PROPOSED
5,paymentFlow,"[고객이 앱을 통해 결제, 결제 정보가 처리됨, 판매자가 수익을 수취]",CONCEPT_GENERATED,REVIEWABLE,PROPOSED
6,personalDataUsage,[],CONCEPT_GENERATED,REVIEWABLE,PROPOSED
7,physicalActivities,"[배송, 포장]",CONCEPT_GENERATED,REVIEWABLE,PROPOSED
8,partnerRequirements,"[지역 농가와의 계약 체결, 식재료의 품질 관리]",CONCEPT_GENERATED,REVIEWABLE,PROPOSED
9,qualificationRequirements,[],CONCEPT_GENERATED,REVIEWABLE,PROPOSED


## 27. Prepared Legal C1 Evidence Summary


In [28]:
legal_adapter = CurrentLegalAdapter() if RUN_STAGED_LEGAL else None
legal_c1_input = (legal_adapter.task_input(candidates[0].candidate, seed)
                  if legal_adapter and candidates else None)
display({'candidateId': candidates[0].candidateId if candidates else None,
         'externalFacts': legal_c1_input['externalFactContext']['facts'] if legal_c1_input else [],
         'note': '공식 근거 수와 allowed index는 Full Legal 응답/실패 diagnostics에서 확인'})

{'candidateId': 'C1',
 'externalFacts': [],
 'note': '공식 근거 수와 allowed index는 Full Legal 응답/실패 diagnostics에서 확인'}

## 28. Full Evidence Judgment — C1 Staged Smoke


In [29]:
RUN_FULL_LEGAL_C1 = RUN_STAGED_LEGAL
legal_one = None
if RUN_FULL_LEGAL_C1 and candidates:
    try:
        legal_one = (await engine.review_legal(seed, candidates[:1]))[0]
        display(show_legal_result([legal_one]))
    except Exception:
        display(show_legal_failure(candidates[0].candidateId, engine.gateway))
else:
    print('SKIPPED — RUN_FULL_LEGAL_C1=True로 명시해야 실행됩니다.')

,candidateId,route,productionStatus,sourceStatus,evidenceCoverage,reviewPhase,factCompletenessStatus,legalSourceStatus,finalEvidenceJudgmentExecuted,recoveryResolution,...,legalClarificationCount,safeSummary,unknownFacts,requiredControls,requiredPartnersAndQualifications,requiredDisclosures,prohibitedVariants,evidenceCount,evidenceRefs,evidenceDiagnostics
0,C1,ACCEPT,IMPLEMENTABLE_WITH_CONTROLS,SOURCE_PARTIAL,"공식 근거를 바탕으로 구현 가능성을 검토했으나, 일부 법률 소스의 조회 범위에는 제...",LEGAL_SOURCE,None,SOURCE_PARTIAL,True,None,...,2,이 사업 모델은 식재료의 품질 관리 및 개인정보 보호 조치를 포함하여 법적으로 구현...,[],"[식재료의 품질 관리 기준을 명확히 설정하고 이를 준수해야 한다., 개인정보 보호를...","[지역 농가와의 계약 체결이 필요하다., 식재료의 품질 관리에 대한 기준을 충족하는...",[고객에게 식재료의 출처 및 품질 관리 기준에 대한 정보를 제공해야 한다.],[],12,"[{'referenceIndex': 0, 'sourceType': 'OFFICIAL...","{'coverageStatus': 'SOURCE_PARTIAL', 'coverage..."


## 29. C1 Route + Staged Redesign/Compliance/Second Legal


In [30]:
c1_portfolio, c1_legal_all, c1_required_inputs, c1_redesigned, c1_replanned = ([], [], [], 0, 0)
if legal_one and candidates:
    c1_portfolio, c1_legal_all, c1_required_inputs, c1_redesigned, c1_replanned = await engine.resolve_legal(
        seed, candidate_preparation.usedPlans, candidates[:1], [legal_one])
display({'initialRoute': legal_one.route.value if legal_one else 'SKIPPED',
         'redesignRequirements': legal_one.redesignRequirements if legal_one else [],
         'recoveryReviews': [item.model_dump(mode='json') for item in c1_legal_all[1:]],
         'requiredInputs': c1_required_inputs, 'terminalCandidates': len(c1_portfolio),
         'redesigned': c1_redesigned, 'replanned': c1_replanned})

{'initialRoute': 'ACCEPT',
 'redesignRequirements': [],
 'recoveryReviews': [],
 'requiredInputs': [],
 'terminalCandidates': 1,
 'redesigned': 0,
 'replanned': 0}

## 30. Remaining 4 Legal + Exhaustive Recovery Summary


In [31]:
RUN_REMAINING_LEGAL = RUN_STAGED_FULL
legal_remaining = []
portfolio, legal_all, required_inputs = (list(c1_portfolio), list(c1_legal_all), list(c1_required_inputs))
redesigned_count, replanned_count = c1_redesigned, c1_replanned
c1_terminal = bool(c1_portfolio or c1_required_inputs or (c1_legal_all and c1_legal_all[-1].route.value == 'SYSTEM_FAILURE'))
if RUN_REMAINING_LEGAL and c1_terminal and len(candidates) > 1:
    legal_remaining = await engine.review_legal(seed, candidates[1:])
    rest_portfolio, rest_legal, rest_inputs, rest_redesigned, rest_replanned = await engine.resolve_legal(
        seed, candidate_preparation.usedPlans, candidates[1:], legal_remaining)
    portfolio += rest_portfolio; legal_all += rest_legal; required_inputs += rest_inputs
    redesigned_count += rest_redesigned; replanned_count += rest_replanned
else:
    print('SKIPPED — C1이 정상 terminal에 도달한 후 remaining Legal을 실행합니다.')
legal_initial = ([legal_one] if legal_one else []) + legal_remaining
display({'recoveryTrace': [item.model_dump(mode='json') for item in legal_all
                           if item.candidateId not in {x.candidateId for x in legal_initial}],
         'requiredInputs': required_inputs, 'metrics': engine._legal_metrics})
print({'Plan Selected': len(selected_plans),
       'Candidate Generated': candidate_preparation.candidateGenerated if candidate_preparation else 0,
       'Candidate Valid Initially': candidate_preparation.candidateAcceptedInitially if candidate_preparation else 0,
       'Candidate Regenerated': candidate_preparation.candidateRegenerated if candidate_preparation else 0,
       'Candidate Recovered': candidate_preparation.candidateRecovered if candidate_preparation else 0,
       'Fact Completion Attempted': legal_preparation.completionAttempted if legal_preparation else 0,
       'Fact Completion Validated': legal_preparation.completionValidated if legal_preparation else 0,
       'Fact Completion Accepted': legal_preparation.completionAccepted if legal_preparation else 0,
       'Dependency Semantic Calls': legal_preparation.dependencySemanticBatchCalls if legal_preparation else 0,
       'Completion Compliance PASS': sum(item.status == 'PASS' for item in legal_preparation.completionCompliance) if legal_preparation else 0,
       'Fact Consistency Invalid': sum(item.status == 'INVALID_FACT' for item in legal_preparation.consistencyReports) if legal_preparation else 0,
       'Consistency Repair Accepted': legal_preparation.consistencyRepairAccepted if legal_preparation else 0,
       'Legal Ready': len(candidates) if legal_preparation else 0,
       'Legal Reviewed': len(legal_initial),
       'Legal Accepted': sum(item.route.value == 'ACCEPT' for item in legal_all),
       'Legal Redesigned': redesigned_count, 'Legal Replanned': replanned_count,
       'Final Portfolio': len(portfolio)})

{'recoveryTrace': [],
 'requiredInputs': [],
 'metrics': {'factAttempted': 0,
  'factValidated': 0,
  'factAccepted': 0,
  'factExhausted': 1,
  'dependencySemanticCalls': 0,
  'completionCompliancePassed': 0,
  'providerNoncompliant': 0,
  'recheckFailed': 0,
  'consistencyInvalid': 1,
  'consistencyRepairAttempted': 1,
  'consistencyRepairAccepted': 0,
  'consistencyRepairExhausted': 1,
  'redesignAttempted': 0,
  'redesignValidated': 0,
  'redesignAccepted': 0,
  'redesignExhausted': 0,
  'replanAttempted': 0,
  'replanValidated': 0,
  'replanAccepted': 0,
  'replanExhausted': 0}}

{'Plan Selected': 5, 'Candidate Generated': 5, 'Candidate Valid Initially': 5, 'Candidate Regenerated': 0, 'Candidate Recovered': 0, 'Fact Completion Attempted': 0, 'Fact Completion Validated': 0, 'Fact Completion Accepted': 0, 'Dependency Semantic Calls': 0, 'Completion Compliance PASS': 0, 'Fact Consistency Invalid': 1, 'Consistency Repair Accepted': 0, 'Legal Ready': 4, 'Legal Reviewed': 4, 'Legal Accepted': 4, 'Legal Redesigned': 0, 'Legal Replanned': 0, 'Final Portfolio': 4}


## 31. Replan


In [32]:
display(show_replan(type('PortfolioView', (), {'concepts': portfolio})()))
print({'replanned': replanned_count, 'reserveAvailable': len(reserve_plans)})

""


{'replanned': 0, 'reserveAvailable': 1}


## 32. Final Portfolio


In [33]:
legal_terminal_status = ('READY_FULL' if len(portfolio) == MAX_CONCEPTS else
    'READY_LIMITED' if portfolio else
    'LEGAL_RECOVERY_COMPLETE_NO_ACCEPTED_CANDIDATE' if legal_initial and len(legal_initial) == len(candidates)
    else 'LEGAL_PENDING')
display(show_final_portfolio(type('PortfolioView', (), {'concepts': portfolio})()) if portfolio else {'status': legal_terminal_status})

,candidateId,lineageId,parentCandidateId,이름,핵심 작동방식,family,descriptor,수익,운영
0,C1,L1,None,소량 맞춤형 식재료 서비스,"식재료를 소량으로 제공하여 낭비를 줄이고, 레시피를 통해 요리의 효율성을 높인다.",거래 중개 · 기타 운영,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 모델을 통해 정기적으로 수익을 창출한다.,주문 후 24시간 이내에 신선한 식재료를 배송한다.
1,C2,L2,None,맞춤형 레시피 추천 서비스,고객의 취향과 재료를 기반으로 한 레시피 추천 시스템을 통해 소량의 식재료를 활용한...,거래 중개 · 파트너 네트워크,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 모델을 통한 수익 창출,"온라인 플랫폼을 통해 레시피를 추천하고, 고객의 피드백을 반영하여 지속적으로 개선한다."
2,C4,L4,None,신선한 재료 정기 배송 서비스,정기 배송 시스템을 통한 신선한 재료 제공으로 고객의 요리 주기에 맞춰 신선한 재료...,거래 중개 · 파트너 네트워크,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 모델을 통한 지속적인 수익 창출,"고객의 피드백을 반영하여 재료의 품질을 개선하고, 지역 농가와 협력하여 신선한 재료..."
3,C5,L5,None,식재료 관리 및 레시피 서비스,소량의 식재료를 활용한 관리 서비스를 통해 음식물 쓰레기를 줄인다.,기타 역할 · 기타 운영,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 모델,온라인 플랫폼을 통해 식재료 관리 서비스를 운영한다.


## 33. Unresolved Candidate Summary


In [34]:
display(show_required_inputs(required_inputs) if required_inputs else {'unresolved': []})

{'unresolved': []}

## 34. Manual Concept Selection


In [35]:
SELECTED_CANDIDATE_ID = portfolio[0].candidateId if portfolio else None  # 사용자가 수정
selected_concept = next((item for item in portfolio if item.candidateId == SELECTED_CANDIDATE_ID), None)
print({'selectedCandidateId': SELECTED_CANDIDATE_ID})

{'selectedCandidateId': 'C1'}


## 35. 7 Hypotheses


In [36]:
hypotheses = (engine.build_or_load_current_hypothesis_contract(selected_concept)
              if RUN_STAGED_FULL and selected_concept else [])
hypotheses = await engine.resolve_hypothesis_semantics(hypotheses) if hypotheses else []
display(show_hypotheses(hypotheses))
display(show_hypothesis_readiness(hypotheses))

,HypothesisType,ProposedValue,FinalValue,SemanticStatus,SemanticReason,Locked,DecisionStatus,LegalImpact
0,TARGET_REGION,대한민국,None,VALID,실제 사업 가설값입니다.,False,PROPOSED,NONE
1,REVENUE_MODEL,구독 모델을 통해 정기적으로 수익을 창출한다.,None,VALID,실제 사업 가설값입니다.,False,PROPOSED,NONE
2,PRICE,"1인 기준 월 30,000원",None,VALID,실제 사업 가설값입니다.,False,PROPOSED,NONE
3,CHANNELS,모바일 앱 및 웹사이트를 통한 주문,None,VALID,실제 사업 가설값입니다.,False,PROPOSED,NONE
4,DIFFERENTIATORS,"고객 맞춤형 레시피 제공, 소량 포장으로 음식물 쓰레기 감소",None,VALID,실제 사업 가설값입니다.,False,PROPOSED,NONE
5,PRE_MARKET_SOM_SHARE,targetSharePercent=5.0 horizonYears=3 rational...,None,VALID,수치·기간·산식 가설이 명시되었습니다.,False,PROPOSED,NONE
6,PRE_MARKET_SOM,amount=5000000.0 currency='KRW' period='연간' ca...,None,VALID,수치·기간·산식 가설이 명시되었습니다.,False,PROPOSED,NONE


{'All Values Semantically Valid': True,
 'All Decisions Confirmed': False,
 'Ready For Handoff': False,
 'status': 'NOT_READY',
 'reason': 'UNRESOLVED_HYPOTHESES',
 'unresolvedHypotheses': ['TARGET_REGION',
  'REVENUE_MODEL',
  'PRICE',
  'CHANNELS',
  'DIFFERENTIATORS',
  'PRE_MARKET_SOM_SHARE',
  'PRE_MARKET_SOM']}

## 36. Confirm / Edit


In [37]:
CONFIRM_ALL_PROPOSED = True
HYPOTHESIS_EDITS = {
    # 'PRICE': '월 17,900원',
}
confirmed_hypotheses = engine.confirm_hypotheses(
    hypotheses, HYPOTHESIS_EDITS, confirm_all_proposed=CONFIRM_ALL_PROPOSED) if hypotheses else []
display(show_hypotheses(confirmed_hypotheses))
hypothesis_readiness = show_hypothesis_readiness(confirmed_hypotheses)
display(hypothesis_readiness)

,HypothesisType,ProposedValue,FinalValue,SemanticStatus,SemanticReason,Locked,DecisionStatus,LegalImpact
0,TARGET_REGION,대한민국,대한민국,VALID,실제 사업 가설값입니다.,False,ACCEPTED,NONE
1,REVENUE_MODEL,구독 모델을 통해 정기적으로 수익을 창출한다.,구독 모델을 통해 정기적으로 수익을 창출한다.,VALID,실제 사업 가설값입니다.,False,ACCEPTED,NONE
2,PRICE,"1인 기준 월 30,000원","1인 기준 월 30,000원",VALID,실제 사업 가설값입니다.,False,ACCEPTED,NONE
3,CHANNELS,모바일 앱 및 웹사이트를 통한 주문,모바일 앱 및 웹사이트를 통한 주문,VALID,실제 사업 가설값입니다.,False,ACCEPTED,NONE
4,DIFFERENTIATORS,"고객 맞춤형 레시피 제공, 소량 포장으로 음식물 쓰레기 감소","고객 맞춤형 레시피 제공, 소량 포장으로 음식물 쓰레기 감소",VALID,실제 사업 가설값입니다.,False,ACCEPTED,NONE
5,PRE_MARKET_SOM_SHARE,targetSharePercent=5.0 horizonYears=3 rational...,targetSharePercent=5.0 horizonYears=3 rational...,VALID,수치·기간·산식 가설이 명시되었습니다.,False,ACCEPTED,NONE
6,PRE_MARKET_SOM,amount=5000000.0 currency='KRW' period='연간' ca...,amount=5000000.0 currency='KRW' period='연간' ca...,VALID,수치·기간·산식 가설이 명시되었습니다.,False,ACCEPTED,NONE


{'All Values Semantically Valid': True,
 'All Decisions Confirmed': True,
 'Ready For Handoff': True,
 'status': 'READY',
 'reason': None,
 'unresolvedHypotheses': []}

## 37. Actual Delta Legal


In [38]:
RUN_DELTA_LEGAL = True
delta_legal_result = None
if RUN_STAGED_FULL and RUN_DELTA_LEGAL and selected_concept and any(h.deltaLegalRequired for h in confirmed_hypotheses):
    delta_legal_result = await engine.review_delta_legal(seed, selected_concept, confirmed_hypotheses)
    confirmed_hypotheses = engine.mark_delta_legal_reviewed(confirmed_hypotheses, delta_legal_result)
display(delta_legal_result.model_dump(mode='json') if delta_legal_result else {'status': 'NOT_REQUIRED_OR_SKIPPED'})

{'status': 'NOT_REQUIRED_OR_SKIPPED'}

## 38. Market Seed


In [39]:
handoff = None
if selected_concept and legal_all and hypothesis_readiness['Ready For Handoff']:
    handoff = engine.build_downstream_handoff(seed, selected_concept, confirmed_hypotheses, legal_all)
display(handoff.marketAnalysisSeedSnapshot if handoff else {
    'status': 'NOT_READY', 'reason': hypothesis_readiness.get('reason'),
    'unresolvedHypotheses': hypothesis_readiness.get('unresolvedHypotheses', [])})

{'contract': 'market-analysis-seed-snapshot-v1',
 'schemaVersion': '2.0',
 'snapshotId': 'lab-market-seed',
 'projectId': 0,
 'selectionId': 0,
 'conceptId': 'C1',
 'createdAt': '2026-08-10T07:58:44.763579+00:00',
 'sourceSnapshotHash': 'sha256:8519c87d0bb3a1f9cdf0ba1fcb68c73d3dff15364aafeb37999a9f958ec3f2ef',
 'originalSeed': {'ideaOverview': '개인 맞춤형 소량 식재료를 제공하고 남은 재료 활용 레시피를 안내하는 서비스',
  'fields': {'ideaOverview': {'value': '개인 맞춤형 소량 식재료를 제공하고 남은 재료 활용 레시피를 안내하는 서비스',
    'source': 'USER_INPUT',
    'decisionState': 'LOCKED'},
   'problem': {'value': '1~2인 가구가 큰 포장 단위 때문에 식재료를 남기고 음식물 쓰레기가 발생한다',
    'source': 'USER_INPUT',
    'decisionState': 'LOCKED'},
   'targetUsers': {'value': '요리할 시간이 적고 낭비를 줄이고 싶은 1~2인 가구',
    'source': 'USER_INPUT',
    'decisionState': 'LOCKED'}}},
 'aiInterpretation': {'interpretedProblem': '1~2인 가구에서 발생하는 음식물 쓰레기 문제를 해결하기 위한 서비스.',
  'interpretedTargetUsers': '요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구.',
  'usageContext': '소량의 식재료를 제공하고 남은 재료를 활용할 수 있는 레시피를 안

## 39. Marketing Source


In [40]:
display(handoff.marketingSourceSnapshot if handoff else {
    'status': 'NOT_READY', 'reason': hypothesis_readiness.get('reason'),
    'unresolvedHypotheses': hypothesis_readiness.get('unresolvedHypotheses', [])})

{'contract': 'marketing-source-snapshot-v1',
 'schemaVersion': '2.0',
 'snapshotId': 'lab-marketing-source',
 'projectId': 0,
 'selectionId': 0,
 'conceptId': 'C1',
 'marketAnalysisSeedSnapshotId': 'lab-market-seed',
 'marketAnalysisSeedSnapshotHash': 'sha256:25d086922da5ac9e28285a65c43752de536792824dc265fa84be6ddd2a544dc1',
 'createdAt': '2026-08-10T07:58:44.763579+00:00',
 'conceptName': '소량 맞춤형 식재료 서비스',
 'targetSegment': '요리할 시간이 적고 낭비를 줄이고 싶은 1~2인 가구',
 'problem': '1~2인 가구가 큰 포장 단위 때문에 식재료를 남기고 음식물 쓰레기가 발생한다.',
 'valueProposition': '필요한 만큼의 식재료를 제공하여 음식물 쓰레기를 줄이고 요리를 간편하게 만든다.',
 'positioning': '개인 맞춤형 소량 식재료와 활용 레시피를 제공하는 서비스.',
 'keyFeatures': ['고객 맞춤형 레시피 제공', '소량 포장으로 음식물 쓰레기 감소', '정기 배송 서비스'],
 'targetRegion': '대한민국',
 'revenueModel': '구독 모델을 통해 정기적으로 수익을 창출한다.',
 'price': '1인 기준 월 30,000원',
 'pricing': '구독 모델을 통해 정기적으로 수익을 창출한다. · 1인 기준 월 30,000원',
 'channels': ['모바일 앱 및 웹사이트를 통한 주문'],
 'competitorDifferentiators': ['고객 맞춤형 레시피 제공, 소량 포장으로 음식물 쓰레기 감소'],
 'preMarketSomShare':

## 40. Contract Compatibility


In [41]:
display(show_downstream_handoff(handoff) if handoff else {
    'contract': 'NOT_READY', 'reason': hypothesis_readiness.get('reason'),
    'unresolvedHypotheses': hypothesis_readiness.get('unresolvedHypotheses', [])})

{'호환성': 'PASS',
 '구조': 'STRUCTURE_PASS',
 '계약': 'CONTRACT_PASS',
 '필드 매핑':                        v2Field                             downstreamField  \
 0        candidate.conceptName        selectedConcept.identity.conceptName   
 1  candidate.solutionMechanism  selectedConcept.solution.solutionMechanism   
 2                hypotheses[*]                             finalHypotheses   
 3                        legal                                 legalResult   
 
               source  transformed  required  
 0  CONCEPT_GENERATED        False      True  
 1  CONCEPT_GENERATED        False      True  
 2     USER_CONFIRMED         True      True  
 3  OFFICIAL_EVIDENCE         True      True  ,
 'Market payload': {'contract': 'market-analysis-seed-snapshot-v1',
  'schemaVersion': '2.0',
  'snapshotId': 'lab-market-seed',
  'projectId': 0,
  'selectionId': 0,
  'conceptId': 'C1',
  'createdAt': '2026-08-10T07:58:44.763579+00:00',
  'sourceSnapshotHash': 'sha256:8519c87d0bb3a1f9cdf0ba

## 41. Trace


In [42]:
display(show_trace(engine.trace))

,순서,시각,stage,action,entity,parent,status,mode,호출,요약,decision,reasonCode
0,1,2026-08-10T07:53:42.186076+00:00,CREATED,CREATED,NaN,NaN,RUNNING,LIVE,NaN,V2 Lab 실행을 생성했습니다.,NaN,NaN
1,2,2026-08-10T07:53:46.809196+00:00,SAFETY_CHECKING,IDEA_BRIEF_DERIVED,NaN,NaN,PASS,LIVE,1.0,Idea interpretation/readiness를 보존했습니다: READY_F...,NaN,NaN
2,3,2026-08-10T07:53:46.809212+00:00,SAFETY_CHECKING,READINESS_INCONSISTENT,NaN,NaN,WARNING,LIVE,NaN,READY_FOR_REVIEW이지만 score=0입니다. V2 gating은 막지 ...,READINESS_INCONSISTENT,NaN
3,4,2026-08-10T07:53:46.911649+00:00,SEED_ANALYZING,ANALYZED,lab-idea-brief,NaN,PASS,LIVE,NaN,필수 3개와 LOCK 3개를 분류했습니다.,NaN,NaN
4,5,2026-08-10T07:53:46.911923+00:00,SEED_ANALYZING,DESIGN_SPACE_READY,NaN,NaN,PASS,LIVE,NaN,Open=11 Constrained=0 Breadth=EXPLORE,NaN,NaN
5,6,2026-08-10T07:53:47.123247+00:00,PLANNING,STARTED,NaN,NaN,RUNNING,LIVE,NaN,최대 5개 동적 plan을 요청합니다.,NaN,NaN
6,7,2026-08-10T07:54:18.421002+00:00,PLANNING,DRAFTS_GENERATED,NaN,NaN,PASS,LIVE,2.0,Plan draft pool=6,NaN,NaN
7,8,2026-08-10T07:54:18.422841+00:00,PLANNING,NORMALIZED,NaN,NaN,PASS,LIVE,NaN,System metadata를 부여한 Plan=6,NaN,NaN
8,9,2026-08-10T07:54:29.613046+00:00,PLANNING,ARCHITECTURE_SEMANTIC_FALLBACK,NaN,NaN,PASS,LIVE,3.0,low-confidence Plan architecture 6개를 batch 분류했...,NaN,NaN
9,10,2026-08-10T07:54:29.613112+00:00,PLAN_VALIDATING,STARTED,NaN,NaN,RUNNING,LIVE,NaN,Opportunity·LOCK·DUPLICATE/VARIANT/DISTINCT 관계...,NaN,NaN


## 42. Provider/Legal Usage


In [43]:
display(show_provider_usage(engine.gateway.usage))
print('상위 외부 작업 수는 내부 AI/MOLEG 네트워크 호출 수와 동일하다고 주장하지 않습니다.')

,논리 작업,상위 외부 작업,논리 stage별,상위 외부 작업 stage별,재시도,소요(ms),모드별,token,보고 비용
0,18,18,"{'SAFETY_CHECKING': 1, 'PLANNING': 1, 'NORMALI...","{'SAFETY_CHECKING': 1, 'PLANNING': 1, 'NORMALI...",0,300756,{'LIVE': 18},None,None


상위 외부 작업 수는 내부 AI/MOLEG 네트워크 호출 수와 동일하다고 주장하지 않습니다.


## 43. Replay Manifest


In [44]:
display(show_replay_manifest(engine.gateway))

{'status': 'REPLAY_PARTIAL',
 'entries':                    operation  \
 0               LEGAL_REVIEW   
 1          SEMANTIC_RELATION   
 2                     EXPAND   
 3    NORMALIZE_ARCHITECTURES   
 4                     EXPAND   
 ..                       ...   
 544    LEGAL_FACT_COMPLETION   
 545    IDEA_BRIEF_DERIVATION   
 546       PLAN_REPLENISHMENT   
 547                PLAN_POOL   
 548             LEGAL_REVIEW   
 
                                                   hash operationVersion  \
 0    00489dd62bff3c4b9e8711093fb3e398a39dde1547de32...               v3   
 1    00c1eeb4f93989a6db3891fb7a22355a71a68800a6f641...               v3   
 2    0120f755ef50e68db6e1d7d1adb0563892b2f762578608...               v3   
 3    013ebfc11d61a317c1041ddcf75902b1dab0d310e73666...               v1   
 4    02bf209b45f806d67ec78f5bdc5e0e24bf99e05fbd297f...               v3   
 ..                                                 ...              ...   
 544  fc333103264d3fbc56f5df30

## 44. One-click MOCK


In [45]:
mock_result = None
if LIVE_TEST_LEVEL == 'FULL_E2E' and MODE != 'LIVE':
    mock_result = await ConceptPortfolioEngine('MOCK').run_full(
        TEST_INPUT, max_concepts=MAX_CONCEPTS, auto_confirm_hypotheses=True)
display(show_run_summary(mock_result) if mock_result else {'status': 'SKIPPED'})
assert mock_result is None or (mock_result.handoff and mock_result.handoff.contractStatus == 'CONTRACT_PASS')

{'status': 'SKIPPED'}

## 45. One-click REPLAY


In [46]:
RUN_ONE_CLICK_REPLAY = False
replay_result = None
if RUN_ONE_CLICK_REPLAY:
    replay_gateway = ProviderGateway('REPLAY', recordings_dir=RECORDINGS_DIR)
    replay_result = await ConceptPortfolioEngine('REPLAY', gateway=replay_gateway).run_full(
        TEST_INPUT, max_concepts=MAX_CONCEPTS, auto_confirm_hypotheses=False)
display(show_run_summary(replay_result) if replay_result else {'status': 'SKIPPED'})

{'status': 'SKIPPED'}

## 46. One-click LIVE


In [47]:
RUN_ONE_CLICK_LIVE = True
live_result = None
if RUN_ONE_CLICK_LIVE and LIVE_TEST_LEVEL == 'ONE_CLICK':
    assert MODE == 'LIVE', 'MODE=LIVE를 먼저 명시하세요.'
    one_click_gateway = ProviderGateway('LIVE', recordings_dir=RECORDINGS_DIR)
    one_click_engine = ConceptPortfolioEngine('LIVE', gateway=one_click_gateway)
    live_result = await one_click_engine.run_full(TEST_INPUT, max_concepts=MAX_CONCEPTS,
                                                  auto_confirm_hypotheses=False)
display(show_run_summary(live_result) if live_result else {'status': 'SKIPPED'})
if live_result:
    display(show_live_validation_summary(LIVE_SCENARIO, live_result))
    display(show_required_inputs(live_result))
    display(show_pre_legal_exclusions(live_result))
    display(show_legal_resolutions(live_result))
    if live_result.runStatus.value == 'FAILED':
        display(show_run_failure(live_result))
        display(show_provider_failure(one_click_engine.gateway))
        display(show_provider_usage(live_result.providerUsage))
        display(show_trace(live_result.trace[-20:]))
        display({'unresolvedCandidates': live_result.unresolvedCandidates,
                 'lastSuccessfulStage': live_result.failureDiagnostics.lastSuccessfulStage if live_result.failureDiagnostics else None,
                 'firstFailedStage': live_result.failureDiagnostics.firstFailedStage if live_result.failureDiagnostics else None})

{'status': 'SKIPPED'}